# Chapter 1: Your First Policy
### *Expected Value*

Welcome to **Pacific Shield Insurance**.

You've been hired as the company's first actuary. Your CEO has landed the company's very first prospective policyholder — a homeowner who needs property coverage.

**Your job: set the annual premium.**

Price it right, and the company profits. Price it too low, and claims eat you alive. Price it too high, and the homeowner walks.

## The math

For each peril, the **expected annual loss** is the frequency (probability of at least one event per year) times the severity (fraction of property value destroyed) times the property value:

$$ \text{expected loss}_i = f_i \cdot s_i \cdot V $$

The **pure premium** is the sum across all perils — the minimum needed to cover expected claims:

$$ P_{\text{pure}} = \sum_i f_i \cdot s_i \cdot V $$

The **gross premium** grosses up for expenses (commissions, overhead, taxes), which are charged as a fraction $e$ of premium:

$$ P_{\text{gross}} = \frac{P_{\text{pure}}}{1 - e} $$

Anything above $P_{\text{gross}}$ is profit margin.

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

## The policy

A $250,000 home with three perils. Expense ratio is 30% (industry-typical for personal lines).

In [ ]:
@dataclass
class Peril:
    name: str
    frequency: float
    severity_pct: float

PERILS = [
    Peril("Fire",  0.020, 0.40),
    Peril("Water", 0.050, 0.10),
    Peril("Theft", 0.005, 0.05),
]

PROPERTY_VALUE     = 250_000
EXPENSE_RATIO      = 0.30
MAX_MARKET_PREMIUM = 8_000

pd.DataFrame([vars(p) for p in PERILS])

## Calculate the pure and gross premium

Before you price the policy, work out the floor.

In [ ]:
def expected_pure_premium(perils, property_value):
    return sum(p.frequency * p.severity_pct * property_value for p in perils)

def expected_gross_premium(perils, property_value, expense_ratio):
    return expected_pure_premium(perils, property_value) / (1 - expense_ratio)

PURE  = expected_pure_premium(PERILS, PROPERTY_VALUE)
GROSS = expected_gross_premium(PERILS, PROPERTY_VALUE, EXPENSE_RATIO)

print(f"Pure premium:  ${PURE:,.2f}   (covers expected claims only)")
print(f"Gross premium: ${GROSS:,.2f}   (covers claims + {EXPENSE_RATIO:.0%} expenses)")

## Simulate a year

Each peril rolls independently. If it fires, the claim is `severity_pct × property_value`.

In [ ]:
def simulate_year(perils, property_value, premium, expense_ratio, rng):
    claims = [
        {"peril": p.name, "amount": property_value * p.severity_pct}
        for p in perils if rng.random() < p.frequency
    ]
    claims_incurred = sum(c["amount"] for c in claims)
    expenses = premium * expense_ratio
    underwriting_income = premium - claims_incurred - expenses
    loss_ratio     = claims_incurred / premium if premium > 0 else 0.0
    combined_ratio = (claims_incurred + expenses) / premium if premium > 0 else 0.0
    return {
        "premium": premium,
        "num_claims": len(claims),
        "claims_incurred": claims_incurred,
        "expenses": expenses,
        "underwriting_income": underwriting_income,
        "loss_ratio": loss_ratio,
        "combined_ratio": combined_ratio,
    }

## Your turn: set the premium

Pick a premium, pick a run length, hit **Run**. The charts show per-year underwriting income and cumulative surplus. With only one policy, expect high year-to-year variance — that's Chapter 2's problem.

In [ ]:
def evaluate(premium, results):
    total_premiums = sum(r["premium"] for r in results)
    total_claims   = sum(r["claims_incurred"] for r in results)
    total_expenses = sum(r["expenses"] for r in results)
    total_income   = total_premiums - total_claims - total_expenses
    avg_loss_ratio = np.mean([r["loss_ratio"] for r in results])
    cum = np.cumsum([r["underwriting_income"] for r in results])
    went_insolvent = bool((cum < 0).any())

    lines = []
    lines.append(f"<p>Over <b>{len(results)}</b> years: "
                 f"earned ${total_premiums:,.0f} in premiums, "
                 f"paid ${total_claims:,.0f} in claims, "
                 f"${total_expenses:,.0f} in expenses.</p>")
    lines.append(f"<p>Net underwriting income: <b>${total_income:,.0f}</b>. "
                 f"Average loss ratio: <b>{avg_loss_ratio:.1%}</b>.</p>")

    if premium < PURE:
        lines.append(f"<p style='color:crimson'><b>FAIL.</b> Premium (${premium:,.0f}) is below the pure premium (${PURE:,.0f}). You didn't even cover expected losses.</p>")
    elif premium < GROSS:
        lines.append(f"<p style='color:darkorange'><b>UNDERPRICED.</b> Premium (${premium:,.0f}) covered losses but not expenses. Break-even is ${GROSS:,.0f}.</p>")
    elif premium > MAX_MARKET_PREMIUM:
        lines.append(f"<p style='color:darkorange'><b>OVERPRICED.</b> ${premium:,.0f} is above the market rate of ${MAX_MARKET_PREMIUM:,.0f}. The homeowner walks.</p>")
    else:
        margin = (premium - GROSS) / premium * 100
        lines.append(f"<p style='color:seagreen'><b>PASS.</b> ${premium:,.0f} covers losses (${PURE:,.0f}), expenses, and includes a {margin:.1f}% profit margin.</p>")

    if went_insolvent:
        lines.append("<p style='color:crimson'>Company went insolvent mid-simulation. With one policy, variance is extreme — a lesson for Chapter 2.</p>")

    lines.append("<hr><p><b>Key takeaway.</b> The pure premium is the floor — the minimum to cover expected claims. The gross premium adds expense loading. Any margin above that is profit.</p>")
    display(HTML("".join(lines)))


premium_slider   = widgets.FloatSlider(value=GROSS * 1.1, min=1_000, max=15_000, step=100,
                                       description="Premium:", readout_format=",.0f",
                                       layout=widgets.Layout(width="500px"))
num_years_slider = widgets.IntSlider(value=10, min=1, max=100, description="Years:",
                                     layout=widgets.Layout(width="500px"))
run_button       = widgets.Button(description="Run simulation", button_style="primary")
output           = widgets.Output()

def run_sim(_):
    premium = premium_slider.value
    n_years = num_years_slider.value
    rng = np.random.default_rng()
    results = [simulate_year(PERILS, PROPERTY_VALUE, premium, EXPENSE_RATIO, rng)
               for _ in range(n_years)]
    df = pd.DataFrame(results)
    df["year"] = df.index + 1
    df["cumulative_income"] = df["underwriting_income"].cumsum()

    with output:
        output.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        colors = np.where(df["underwriting_income"] >= 0, "seagreen", "crimson")
        axes[0].bar(df["year"], df["underwriting_income"], color=colors)
        axes[0].axhline(0, color="black", linewidth=0.5)
        axes[0].set_title("Underwriting income by year")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("$")
        axes[1].plot(df["year"], df["cumulative_income"], marker="o")
        axes[1].axhline(0, color="black", linewidth=0.5)
        axes[1].set_title("Cumulative surplus")
        axes[1].set_xlabel("Year")
        axes[1].set_ylabel("$")
        plt.tight_layout()
        plt.show()

        display(df[["year", "num_claims", "claims_incurred", "expenses",
                    "underwriting_income", "loss_ratio", "combined_ratio",
                    "cumulative_income"]].style.format({
            "claims_incurred":     "${:,.0f}",
            "expenses":            "${:,.0f}",
            "underwriting_income": "${:,.0f}",
            "loss_ratio":          "{:.1%}",
            "combined_ratio":      "{:.1%}",
            "cumulative_income":   "${:,.0f}",
        }))
        evaluate(premium, results)

run_button.on_click(run_sim)
display(widgets.VBox([premium_slider, num_years_slider, run_button, output]))

## Hints

<details><summary>Hint 1 — where to start</summary>

Think about the expected cost of each peril separately. What's the average loss from fire in any given year?

</details>

<details><summary>Hint 2 — the formula</summary>

Expected loss per peril = probability × severity × property value. Calculate this for each peril and add them up.

</details>

<details><summary>Hint 3 — the numbers</summary>

```
Fire  = 0.020 × 0.40 × $250,000 = $2,000.00
Water = 0.050 × 0.10 × $250,000 = $1,250.00
Theft = 0.005 × 0.05 × $250,000 = $   62.50
Pure premium total              = $3,312.50
```

</details>

<details><summary>Hint 4 — expense loading</summary>

The pure premium only covers expected claims. 30% of premium goes to expenses, so:

```
gross premium = pure_premium / (1 − 0.30)
              = $3,312.50 / 0.70
              = $4,732.14
```

Add a profit margin on top.

</details>

## What's next

**Chapter 2 — Growing the Book (Law of Large Numbers).** You priced one policy well, but one policy is a coin flip. In Chapter 2 you'll write thousands of policies and watch how volume tames that volatility — the foundation the whole industry is built on.